# Create a Content Understanding Analyzer

This notebook shows how to create a custom **Azure AI Content Understanding** analyzer that extracts structured fields from documents.

An *analyzer* is a reusable AI pipeline. Once created, you can reuse it to process many documents without redefining the extraction schema.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv requests --quiet

In [ ]:
import os
import json
import requests
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_AI_ENDPOINT"].rstrip("/")
credential = DefaultAzureCredential()

# Acquire a bearer token for the Cognitive Services scope
token = credential.get_token("https://cognitiveservices.azure.com/.default").token
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

API_VERSION = "2025-11-01"
print(f"Endpoint: {endpoint}")
print("Authentication ready.")

In [ ]:
# Define the analyzer schema.
# This example extracts invoice-related fields from PDF documents.
ANALYZER_ID = "invoiceAnalyzerDemo"

analyzer_definition = {
    "description": "Extracts key fields from invoices",
    "baseAnalyzerId": "prebuilt-document",
    "models": {
        "completion": "gpt-4.1",
        "embedding": "text-embedding-3-large"
    },
    "fieldSchema": {
        "fields": {
            "InvoiceId": {
                "type": "string",
                "method": "extract",
                "description": "The unique invoice number"
            },
            "VendorName": {
                "type": "string",
                "method": "extract",
                "description": "Name of the vendor or supplier"
            },
            "InvoiceDate": {
                "type": "date",
                "method": "extract",
                "description": "Date the invoice was issued"
            },
            "TotalAmount": {
                "type": "number",
                "method": "extract",
                "description": "Total amount due on the invoice"
            }
        }
    }
}

print(f"Analyzer ID: {ANALYZER_ID}")
print("Definition prepared.")

In [ ]:
# Create (or update) the analyzer via the REST API
url = f"{endpoint}/contentunderstanding/analyzers/{ANALYZER_ID}?api-version={API_VERSION}"

response = requests.put(url, headers=headers, json=analyzer_definition)

if response.status_code == 409:
    # Analyzer already exists — fetch its current state instead
    print("Analyzer already exists. Fetching current state...")
    response = requests.get(url, headers=headers)
    response.raise_for_status()
elif not response.ok:
    print(f"Error {response.status_code}:")
    print(response.text)
    response.raise_for_status()

result = response.json()
print("Analyzer created / updated:")
print(json.dumps(result, indent=2))

In [ ]:
# List all analyzers in the resource
list_url = f"{endpoint}/contentunderstanding/analyzers?api-version={API_VERSION}"

list_response = requests.get(list_url, headers=headers)
list_response.raise_for_status()

analyzers = list_response.json().get("value", [])
print(f"Total analyzers found: {len(analyzers)}")
for a in analyzers:
    print(f"  - {a.get('analyzerId', a.get('id', 'unknown'))}")

In [ ]:
# Optional cleanup — delete the analyzer when done
# Uncomment the lines below to delete the demo analyzer

# delete_url = f"{endpoint}/contentunderstanding/analyzers/{ANALYZER_ID}?api-version={API_VERSION}"
# delete_response = requests.delete(delete_url, headers=headers)
# delete_response.raise_for_status()
# print(f"Analyzer '{ANALYZER_ID}' deleted.")

print("Skipping cleanup — analyzer preserved for use in 02_analyze_document.ipynb.")